# torch.distributed.fsdp
- FSDP2（Fully Sharded Data Parallel v2）是 PyTorch 官方推出的原生分布式大模型训练框架，是对传统 FSDP1 的全面重写与重大升级。
- 作为 PyTorch 团队主推的下一代标准方案，<font color='red'>它的核心目标是解决大模型训练中“单卡显存不足”和“分布式代码改造成本高”两大痛点，</font>
- 在保持极低代码侵入性的同时，实现对标甚至超越 DeepSpeed ZeRO 和 Megatron-LM 的性能与扩展性。

## 、 核心架构演进：从 FlatParameter 到 DTensor
<font color='red'>FSDP2 最底层的重构在于彻底移除了 FSDP1 中的 FlatParameter 设计，转而采用 DTensor 作为参数分片的底层载体。</font>

- 消除通信开销：<font color='red'>基于 DTensor 的设计使得保存和加载检查点时，可以直接获取无通信的分片状态字典（Communication-free sharded state_dict），大幅提升了断点续训的效率。</font>

- <font color='red'>确定性内存管理</font>：避免了旧版本中因 recordStream 导致的内存碎片化问题，使大规模训练的显存占用更加稳定可控。
- <font color='red'> 灵活的参数控制</font>：打破了 FlatParameter 将多个参数打包带来的灵活性限制，支持对单个参数进行冻结、转换等更精细的操作。

## 二、 核心特性与多维并行能力
- 四维并行无缝组合：
  - FSDP2 不再局限于单一的数据并行，而是原生支持 DP（数据并行）、TP（张量并行）、PP（流水线并行）以及 EP（专家并行）的自由组合，能够覆盖千亿/万亿级超大模型的所有训练场景。
- 深度兼容 torch.compile：
  - FSDP2 完美支持 PyTorch 的编译模式。通过联合图优化（Joint-Graph Optimization），<font color='red'>它能在编译模式下自动发现最佳的参数分桶策略（auto-bucketing），最大化计算与通信的重叠，消除 graph break，</font>显著提升训练吞吐量。
- 先进的混合精度支持：
  - 原生支持 FP8 混合精度训练，能够在保证精度的前提下减少通信带宽和显存占用，相比 BF16 训练可带来 30%—50% 的吞吐提升。
- 高级 CPU Offloading：
  - 改进了参数和优化器状态的卸载机制，特别是实现了 CPU offloading 与梯度积累的完美结合，使得在内存受限的环境中训练更大规模的模型成为可能。

## 三、 性能优势与实测表现
在实际应用中，FSDP2 展现出了显著的性能红利。以强化学习框架 verl 针对 Qwen2.5-7B 模型的测试为例，相较于传统的 FSDP1，FSDP2 结合智能 offloading 策略后：
- 吞吐量提升：tokens/s 提升了约 50%。
- 内存占用降低：峰值显存占用减少了约 21%。
- 通信开销缩减：通信带宽消耗降低了约 29%。

此外，在 TorchTitan 库的 8x H100 环境下，FSDP2 相比传统 FSDP 实现了 7% 的内存节省和更高的模型 FLOPS 利用率（MFU），且保持了完全一致的损失收敛曲线。

## 四、 总结
- 如果说 FSDP1 是 PyTorch 探索全分片数据并行的试水之作，
- 那么 FSDP2 则是集大成者。它不仅继承了 PyTorch 原生生态“易用、低侵入”的基因，还补齐了现代大模型训练所需的极致内存优化、高效通信和编译加速能力，是当前构建大规模 AI 基础设施的首选分布式训练范式。

## FSDP2
- 以下为您提供 FSDP2 在 PyTorch 中的标准代码实现示例。
- 该示例展示了 FSDP2 核心的“元设备初始化（Meta Device Init）”流程、分层分片策略以及混合精度配置：

In [ ]:
import torch
from torch.distributed.fsdp import fully_shard, MixedPrecisionPolicy

# 1. 在元设备（Meta Device）上创建模型，避免在单卡上实例化完整模型导致显存溢出
with torch.device("meta"):
    model = Transformer() 

# 2. 定义混合精度策略
mp_policy = MixedPrecisionPolicy(
    param_dtype=torch.bfloat16,    # 前向与反向计算使用 bfloat16
    reduce_dtype=torch.float32     # 梯度规约时使用 float32 以保持精度
)

# 3. 采用显式分层分片（FSDP2 移除了 auto_wrap_policy，推荐以 TransformerBlock 为单位）
for module in model.modules():
    if isinstance(module, TransformerBlock):
        fully_shard(module, mp_policy=mp_policy)

# 对整体模型应用分片
fully_shard(model, mp_policy=mp_policy)

# 4. 将张量实体化到 GPU 上分配空间（此时仅在每张卡上分配 1/N 的显存）
model.to_empty(device="cuda")

# 5. 执行用户定义的权重初始化
model.init_weights()  

# 6. 初始化优化器（注意：必须在 fully_shard 之后进行）
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

## 代码核心要点解析：
- API 变更：FSDP2 摒弃了 FSDP1 中通过类包装的方式，改用更简洁的 fully_shard 函数。
- 内存管理：通过 torch.device("meta") 和 to_empty(device="cuda") 的组合，彻底避免了传统方式下先在 CPU/GPU 加载完整模型再切分带来的巨大内存峰值开销。
- 原生参数访问：由于底层采用了 DTensor 架构，分片后的参数可以直接作为独立的张量进行操作，无需像旧版本那样解包 FlatParameter。

# 比较FlatParameter和DTensor

- 从 FSDP1 的 FlatParameter 演进到 FSDP2 的 DTensor，是 PyTorch 在分布式大模型训练底层架构上的一次彻底重构。
- 这一转变不仅仅是 API 的简化，<font color='red'>更是为了解决旧版设计在灵活性、内存管理和生态兼容性上的根本痛点。</font>

以下是这两者在核心设计与工程表现上的详细对比：
## 1. 参数表示与切分机制
- FlatParameter (FSDP1)：采用“扁平化缓冲区”设计。
  - 它将一个 FSDP unit（例如整个 Transformer Block）内的所有参数（如 Q/K/V 投影矩阵、LayerNorm 权重等）展平并拼接成一个巨大的 1D Tensor，然后对这个 1D Tensor 进行切分和通信。
  - 这种设计主要是为了迎合 NCCL 后端对集合通信操作的性能要求。
  
- DTensor (FSDP2)：采用“逐参数分片（Per-parameter sharding）”设计。
  - <font color='red'>每个 nn.Parameter 都被独立地替换为一个携带额外元数据的 DTensor</font>，并在第 0 维（dim-0）上进行物理切分。
  - 它不再将多个参数强行打包，而是保留了原始张量的结构信息。
  
## 2. 灵活性与可组合性
- FlatParameter (FSDP1)：由于多个参数被揉合在一个不透明的扁平缓冲区中，导致无法对单个参数执行差异化操作。
  - 例如，想要冻结某一层特定的参数、或者对个别参数应用不同的类型转换变得极其复杂，严重降低了组件的可组合性。
- DTensor (FSDP2)：原生支持对单个参数的便捷操作。
  - 开发者可以像使用普通张量一样直接访问和操作分片后的参数，极大地提升了代码编写的灵活性。
  
## 3. 状态字典（State Dict）与检查点管理
- FlatParameter (FSDP1)：保存和加载检查点时心智负担较重。
  - <font color='red'>由于参数已经被展平，恢复时需要复杂的逻辑来重新映射键名（FQN），且通常需要额外的通信开销来对齐状态。</font>
- DTensor (FSDP2)：<font color='red'>实现了无通信的分片状态字典（Communication-free sharded state_dict）</font>。
  - <font color='red'>因为每个参数都有独立的 DTensor 表示</font>，保存和加载检查点可以直接获取本地分片数据，无需触发昂贵的跨节点通信。同时，模型的完全限定名称（FQN）保持不变，保持了代码的一致性。
  
## 4. 内存管理与确定性
- FlatParameter (FSDP1)：依赖 CUDA 缓存分配器和 record_stream 机制来跟踪跨流张量的生命周期。
  - <font color='red'>当通信流与计算流交替使用同一物理内存块时，容易引发偶发性的竞态条件，导致显存峰值波动甚至 OOM（内存溢出）。</font>
- DTensor (FSDP2)： OOM（内存溢出）引入了更优的动态内存池管理系统</font>，<font color='green'>避免了对 record_stream 的依赖且无需 CPU 同步。</font>
    - 内存泄漏（慢性问题）：程序中已经不再使用的对象（垃圾），由于被错误地持续引用，导致垃圾回收器（GC）无法将其清理。这些垃圾慢慢堆满房间，挤占了新家具的空间。
    - 内存溢出（急性结果）：内存需求大于可用空间，直接抛出 OOM 错误。它可能是因为长期的内存泄漏累积导致的，也可能是因为程序本身逻辑不合理（如一次性加载超大文件、无限创建对象等）引发的。
  - 这使得 GPU 内存占用不仅更低，而且具有高度的确定性。实测表明，FSDP2 相比 FSDP1 能降低约 7% 的峰值内存占用。
  
## 5. 与现代编译器的兼容度
- FlatParameter (FSDP1)：由于其内部的不透明封装，在与 torch.compile 结合时极易产生 Graph Break（图断裂），阻碍了编译器进行全图优化。
- DTensor (FSDP2)：作为 PyTorch 官方的分布式标准抽象，<font color='red'>它与 torch.compile 深度兼容</font>。通过联合图优化（Joint-Graph Optimization），能够消除 graph break，从而最大化提升训练速度。

## 总结：
- FlatParameter 是早期为了追求极致通信性能而做出的妥协，牺牲了灵活性；
- 而 DTensor 则是在现代硬件和编译器加持下，兼顾了高性能、高灵活性和易用性的新一代分布式张量基石。